In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def earth_mover_distance_loss(logits: torch.Tensor, true_labels: torch.Tensor) -> torch.Tensor:
    """
    Calculates the Earth Mover's Distance (EMD) loss for ordinal classification.

    This loss assumes classes have a natural ordering. It measures the L1 distance
    between the Cumulative Distribution Function (CDF) of the predicted probabilities
    and the CDF of the true labels.

    Args:
        logits: Raw output scores from the model (before softmax).
                Shape: (batch_size, num_classes)
        true_labels: Ground truth class indices.
                     Shape: (batch_size,)
                     Values should be integers from 0 to num_classes - 1.

    Returns:
        torch.Tensor: The mean EMD loss over the batch.
    """
    num_classes = logits.shape[1]
    batch_size = logits.shape[0]

    # 1. Convert logits to probabilities
    pred_probs = F.softmax(logits, dim=1)

    # 2. Calculate predicted CDF
    # cumsum computes the cumulative sum along a given dimension
    pred_cdf = torch.cumsum(pred_probs, dim=1)
    # Shape: (batch_size, num_classes)

    # 3. Create true label distribution (one-hot encoding)
    # Ensure true_labels are long type for one_hot
    true_labels_long = true_labels.long()
    # Check for out-of-bounds labels
    if torch.any(true_labels_long < 0) or torch.any(true_labels_long >= num_classes):
        raise ValueError(f"true_labels contain values out of range [0, {num_classes-1}]")

    true_dist = F.one_hot(true_labels_long, num_classes=num_classes).float()
    # Shape: (batch_size, num_classes)

    # 4. Calculate true label CDF
    true_cdf = torch.cumsum(true_dist, dim=1)
    # Shape: (batch_size, num_classes)

    # 5. Calculate EMD (L1 distance between CDFs)
    # Sum the absolute differences along the class dimension
    emd = torch.sum(torch.abs(pred_cdf - true_cdf), dim=1)
    # Shape: (batch_size,)

    # 6. Average the loss over the batch
    mean_emd = torch.mean(emd)

    return mean_emd





In [2]:

# Configuration
batch_size = 4
num_classes = 5 # Example: Ordinal classes like [1_star, 2_star, 3_star, 4_star, 5_star]

# Dummy model output (logits)
# Simulate raw scores from a neural network
dummy_logits = torch.randn(batch_size, num_classes, requires_grad=True)
print("Dummy Logits:\n", dummy_logits)

# Dummy true labels (class indices)
# Must be within [0, num_classes - 1]
dummy_true_labels = torch.tensor([0, 4, 2, 1]) # Example true ratings
print("\nTrue Labels:\n", dummy_true_labels)

# Calculate the EMD loss
loss = earth_mover_distance_loss(dummy_logits, dummy_true_labels)
print(f"\nCalculated EMD Loss: {loss.item():.4f}")

# Example of how to use it in a backward pass (requires_grad=True for logits)
if dummy_logits.requires_grad:
    loss.backward()
    print("\nGradients w.r.t. Logits (first element):\n", dummy_logits.grad[0])

# --- Example showing penalty difference ---
print("\n--- Penalty Difference Example ---")
# Case 1: Prediction close to true label
logits_close = torch.tensor([[10.0, 1.0, -1.0, -2.0, -3.0]], requires_grad=True) # Strong prediction for class 0
true_label_close = torch.tensor([0])
loss_close = earth_mover_distance_loss(logits_close, true_label_close)
print(f"Logits: {logits_close.detach().numpy()}, True Label: {true_label_close.item()}, Loss: {loss_close.item():.4f}")

# Case 2: Prediction far from true label
logits_far = torch.tensor([[-3.0, -2.0, -1.0, 1.0, 10.0]], requires_grad=True) # Strong prediction for class 4
true_label_far = torch.tensor([0]) # Same true label as above
loss_far = earth_mover_distance_loss(logits_far, true_label_far)
print(f"Logits: {logits_far.detach().numpy()}, True Label: {true_label_far.item()}, Loss: {loss_far.item():.4f}")

# Case 3: Prediction adjacent to true label
logits_adj = torch.tensor([[1.0, 10.0, -1.0, -2.0, -3.0]], requires_grad=True) # Strong prediction for class 1
true_label_adj = torch.tensor([0]) # Same true label as above
loss_adj = earth_mover_distance_loss(logits_adj, true_label_adj)
print(f"Logits: {logits_adj.detach().numpy()}, True Label: {true_label_adj.item()}, Loss: {loss_adj.item():.4f}")

# Observe that loss_far > loss_adj > loss_close, demonstrating the ordinal penalty
assert loss_far > loss_adj > loss_close, "EMD loss should penalize distant errors more"
print("Loss comparison confirms EMD penalizes distant errors more.")




Dummy Logits:
 tensor([[-0.9455, -0.4552,  0.4102, -0.6752, -2.2894],
        [ 1.4670,  0.3538, -0.2026,  1.4993, -0.7891],
        [ 1.6788, -0.7886,  1.2492, -0.0176, -1.7962],
        [ 0.4445,  0.9594,  0.0127, -0.3369, -0.0370]], requires_grad=True)

True Labels:
 tensor([0, 4, 2, 1])

Calculated EMD Loss: 1.5965

Gradients w.r.t. Logits (first element):
 tensor([-0.0550, -0.0392,  0.0267,  0.0496,  0.0179])

--- Penalty Difference Example ---
Logits: [[10.  1. -1. -2. -3.]], True Label: 0, Loss: 0.0002
Logits: [[-3. -2. -1.  1. 10.]], True Label: 0, Loss: 3.9998
Logits: [[ 1. 10. -1. -2. -3.]], True Label: 0, Loss: 0.9999
Loss comparison confirms EMD penalizes distant errors more.


In [1]:
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
import sys, os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import torch.multiprocessing as mp
import torchvision
import torchvision.transforms as transforms
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import Dataset
import torch.optim as optim
import pickle as pk
import sys, os
sys.path.append('/projects/bdne/spandey3/halo_gotham/GOTHAM/src')
from model_enc_dec import *
import numpy as np
import h5py as h5
import torch
from torch.nn import functional as F
from dataclasses import dataclass
from contextlib import nullcontext
from dataclasses import dataclass
from torch.nn.parallel import DistributedDataParallel as DDP
from multiprocessing import Pool
import ast
%load_ext autoreload
%autoreload 2

import argparse

def parse_args():
    parser = argparse.ArgumentParser(description='Training script with key-value arguments')
    
    # Define arguments with their default values and types
    parser.add_argument('--grid_sbox', type=int, default=8, 
                        help='Grid size parameter')
    parser.add_argument('--add_space_token', type=lambda x: x.lower() == 'true', 
                        default=False, help='Whether to add space token')
    parser.add_argument('--subsel_type', type=str, default='all',
                        help='Subset selection type')
    parser.add_argument('--learning_rate', type=float, default=5e-4,
                        help='Learning rate')
    parser.add_argument('--max_iters', type=int, default=250,
                        help='Maximum iterations')
    parser.add_argument('--loss_type', type=str, default='EMD',
                        help='Maximum iterations')    
    args = parser.parse_args()
    return args


In [2]:
# args = parse_args()



In [3]:

# if __name__ == "__main__":
# args = parse_args()
# grid_sbox = args.grid_sbox
# add_space_token = args.add_space_token
# subsel_type = args.subsel_type
# learning_rate = args.learning_rate
# max_iters = args.max_iters
# loss_type = args.loss_type

# try:
#     grid_sbox = int(ast.literal_eval(sys.argv[-5]))
#     add_space_token = bool(ast.literal_eval(sys.argv[-4]))
#     subsel_type = sys.argv[-3]
#     learning_rate = float(ast.literal_eval(sys.argv[-2]))
#     max_iters = int(ast.literal_eval(sys.argv[-1]))
# except:
grid_sbox = 8
add_space_token = False
subsel_type = 'all'    
learning_rate = 5e-4
max_iters = 250
loss_type = 'EMD'
print(f"grid_sbox = {grid_sbox}, add_space_token = {add_space_token}, subsel_type = {subsel_type}, learning_rate = {learning_rate}, max_iters = {max_iters}, loss_type = {loss_type}")

# print(learning_rate, max_iters)

def setup(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()


# def train():
device = 'cuda'
compile = True 
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True 
device_type = 'cuda'
dtype = 'bfloat16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# dist.init_process_group("nccl")
rank = 0
print(f"Start running basic DDP example on rank {rank}.")
# Ndevices = torch.cuda.device_count()
# Ndevices = 16
Ndevices = 1

BoxSize = 1000.
grid = 32
# grid_sbox = 32
nvocab = 64
nrand_sel_box = 512
subsamp_ds = 1
ds_fac_here = 8
# ds_type_here = 'random'
ds_type_here = 'seq'
# rand_seed_dsfac = 0
rand_seed_dsfac = 1
# add_space_token = False
# Mstar_cut = 8.5
Mstar_cut = 12.7

torch.cuda.empty_cache()
device_id = rank % torch.cuda.device_count()
sdir = '/work/hdd/bdne/spandey3/quijote_data/halo_gotham_data/process_split'
savefname = f'{sdir}/SPLIT_DMO_DATA_{Ndevices}_gpus_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}.h5'
# dist.barrier()
with h5.File(savefname, 'r') as f:
    ind_all_train = np.arange(f[f'dm_train_dev_{rank}'][:].shape[0])
    ind_all_val = np.arange(f[f'dm_val_dev_{rank}'][:].shape[0])
    if ds_type_here == 'random':
        np.random.seed(rand_seed_dsfac)
        ind_all_train = np.random.permutation(ind_all_train)
        ind_all_val = np.random.permutation(ind_all_val)
        ind_sel_train = ind_all_train[::ds_fac_here]
        ind_sel_val = ind_all_val[::ds_fac_here]
    else:
        ind_sel_train = ind_all_train[rand_seed_dsfac::ds_fac_here]
        ind_sel_val = ind_all_val[rand_seed_dsfac::ds_fac_here]

    print(f"ind_sel_train = {ind_sel_train.shape}, ind_sel_val = {ind_sel_val.shape}", flush=True)
    dm_train_gpu = torch.tensor(f[f'dm_train_dev_{rank}'][:][ind_sel_train]).to(ptdtype).to(device_id, non_blocking=True)
    dm_val_gpu = torch.tensor(f[f'dm_val_dev_{rank}'][:][ind_sel_val]).to(ptdtype).to(device_id, non_blocking=True)
    grid_size = int(f['grid'][()])
f.close()
# dist.barrier()
torch.cuda.empty_cache()


savefname = f'{sdir}/SPLIT_HALO_DATA_{Ndevices}_gpus_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_spacetoken_{add_space_token}_xMvc_{Mstar_cut}.h5'
# dist.barrier()
with h5.File(savefname, 'r') as f:
    x_train_gpu = torch.tensor(f[f'x_train_dev_{rank}'][:][ind_sel_train]).to(torch.long).to(device_id, non_blocking=True)
    y_train_gpu = torch.tensor(f[f'y_train_dev_{rank}'][:][ind_sel_train]).to(torch.long).to(device_id, non_blocking=True)
    mask_train_gpu = torch.tensor(f[f'mask_train_dev_{rank}'][:][ind_sel_train]).to(ptdtype).to(device_id, non_blocking=True)
    params_train_gpu = torch.tensor(f[f'params_train_dev_{rank}'][:][ind_sel_train]).to(ptdtype).to(device_id, non_blocking=True)

    x_val_gpu = torch.tensor(f[f'x_val_dev_{rank}'][:][ind_sel_val]).to(torch.long).to(device_id, non_blocking=True)
    y_val_gpu = torch.tensor(f[f'y_val_dev_{rank}'][:][ind_sel_val]).to(torch.long).to(device_id, non_blocking=True)
    mask_val_gpu = torch.tensor(f[f'mask_val_dev_{rank}'][:][ind_sel_val]).to(ptdtype).to(device_id, non_blocking=True)
    params_val_gpu = torch.tensor(f[f'params_val_dev_{rank}'][:][ind_sel_val]).to(ptdtype).to(device_id, non_blocking=True)

    nvocab_total = f['nvocab_total'][()]
    start_token = f['start_token'][()]
    pad_token = int(f['pad_token'][()])
    end_token = f['end_token'][()]
    max_sentence_length = f['max_sentence_length'][()]  
f.close()
# dist.barrier()
torch.cuda.empty_cache()

indices = torch.arange(dm_train_gpu.shape[1])

dm_train_gpu = dm_train_gpu[:,indices,...]
dm_val_gpu = dm_val_gpu[:,indices,...]

print(subsel_type, indices, dm_train_gpu.shape, dm_val_gpu.shape, add_space_token)

# max_iters = 3000
eval_interval = 10
# learning_rate = 3e-4
# max_iters = 1500
eval_iters = 8
n_embd = 256
# n_head = 8
# n_layer = 8

n_head = 8
n_layer = 4

dropout = 0.2
nparams = 5 # number of parameters in camels to append to the CNN features output
vocab_size = nvocab_total
block_size = max_sentence_length - 1
print(f"block_size = {block_size}, vocab_size = {vocab_size}, pad_token = {pad_token}, max_sentence_length = {max_sentence_length}")
print(f"nembd = {n_embd}, nhead = {n_head}, nlayer = {n_layer}, nparams = {nparams}, dropout = {dropout}")

if grid_sbox == 32:
    layers_types =  ['res', 'res', 'res', 'res']
if grid_sbox == 16:
    layers_types =  ['res', 'res', 'res']
if grid_sbox == 8:
    layers_types =  ['res','res']
    
    
HaloConfig = {'block_size': block_size, 'vocab_size': vocab_size, 'n_layer': n_layer, 
                'n_head': n_head, 'n_embd': n_embd, 'nparams': nparams, 'dropout': dropout, 
                'bias': True, 'ksize': 3, 'density_grid_in': grid_size, 'density_grid_out': 4, 
                'ninp_density': dm_train_gpu.shape[1], 'pad_token': pad_token, 'flash': True,
                'dmo_cond_embed_type':'vit', 'layers_types':layers_types,
                'n_layers_vit': 2, 'n_heads_vit': 8}


model = HaloDecoderModel(HaloConfig).to(device_id)

# load the model checkpoint:

# cp_name = f'/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL4_fidfinetune_TEST_model_hres_encdec_ddp_grid_8_nvocab_64_nembed_256_nhead_8_nrandsubsel_8192_subselDMOfields_all_Mstarcut_12.7_spacetoken_False_maxiter_1500_lr_0.0005.pt'
# cp_name = '/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL3_TEST_model_hres_encdec_ddp_grid_8_nvocab_64_nembed_256_nhead_8_nrandsubsel_2048_subselDMOfields_all_Mstarcut_12.7_spacetoken_False_maxiter_1500_lr_0.0005.pt'
# cp_name = '/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL5_nofidfinetune_seq_TEST_model_hres_encdec_ddp_grid_8_nvocab_64_nembed_256_nhead_8_nrandsubsel_2048_subselDMOfields_all_Mstarcut_12.7_spacetoken_False_maxiter_1500_lr_0.0005.pt'

# cp_name = '/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL_TEST_model_hres_encdec_ddp_grid_8_nvocab_64_nembed_256_nhead_8_nrandsubsel_2048_subselDMOfields_all_Mstarcut_12.7_spacetoken_False_maxiter_1500_lr_0.0005.pt'
# cp_name = '/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/iter_save/FINAL2_iter_124_seq_TEST_model_hres_encdec_ddp_grid_8_nvocab_64_nembed_256_nhead_8_nrandsubsel_2048_subselDMOfields_all_Mstarcut_12.7_spacetoken_False_maxiter_1500_lr_0.0005.pt'

# checkpoint = torch.load(cp_name, map_location=f'cuda:{device_id}')    
# model.load_state_dict(checkpoint['model'])

if rank == 0: print(f"Init model and loaded to GPU", flush=True)            
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# model = DDP(model, device_ids=[device_id])
# model = 
model = torch.nn.DataParallel(model)



def get_batch(split, ji=0, batch_size=None):
    if split == 'train':
        x = x_train_gpu
        y = y_train_gpu
        mask = mask_train_gpu
        dm = dm_train_gpu
        params = params_train_gpu

    elif split == 'val':
        x = x_val_gpu
        y = y_val_gpu
        mask = mask_val_gpu
        dm = dm_val_gpu
        params = params_val_gpu

    if batch_size is not None:
        x = x[batch_size*(ji):batch_size*(ji+1)].to(device_id, non_blocking=True)
        y = y[batch_size*(ji):batch_size*(ji+1)].to(device_id, non_blocking=True)
        mask = mask[batch_size*(ji):batch_size*(ji+1)].to(device_id, non_blocking=True)
        dm = dm[batch_size*(ji):batch_size*(ji+1)].to(device_id, non_blocking=True)
        params = params[batch_size*(ji):batch_size*(ji+1)].to(device_id, non_blocking=True)

    return x, y, mask, dm, params

# helps estimate an arbitrarily accurate loss over either split using many batches
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y, MASK, DM, PARAMS = get_batch(split, batch_size = batch_size)
            with ctx:
                logits, loss = model(X, DM, params=PARAMS, maskd=MASK, targets=Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    return out    

decay_lr = True # whether to decay the learning rate
decay_lr_model = 'cosine'
warmup_iters = 100 # how many steps to warm up for
lr_decay_iters = max_iters # should be ~= max_iters per Chinchilla
min_lr = learning_rate/10. # minimum learning rate, should be ~= learning_rate/10 per Chinchilla
# learning rate decay scheduler (cosine with warmup)
def get_lr(it, model='cosine'):
    # 1) linear warmup for warmup_iters steps
    if model == 'cosine':
        if it < warmup_iters:
            return learning_rate * it / warmup_iters
        # 2) if it > lr_decay_iters, return min learning rate
        if it > lr_decay_iters:
            return min_lr
        # 3) in between, use cosine decay down to min learning rate
        decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
        assert 0 <= decay_ratio <= 1
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges 0..1
        return min_lr + coeff * (learning_rate - min_lr)
    
    elif model == 'linear':
        if it < warmup_iters:
            return learning_rate * it / warmup_iters
        else:
            return learning_rate - (it - warmup_iters) * (learning_rate - min_lr) / (lr_decay_iters - warmup_iters)

    elif model == 'constant':
        return learning_rate



iter_num = 0
local_iter_num = 0 # number of iterations in the lifetime of this process
running_mfu = -1.0    
best_val_loss = 1e20
# nbatches = 64
# batch_size = 320
batch_size = 1024
# batch_size = 768
nbatches = len(x_train_gpu) // batch_size
print(f"nbatches = {nbatches}, total train size = {len(x_train_gpu)}")

eval_interval = 4
save_separate_interval = 10

# accumulation_steps = 1  # Accumulate gradients over 2 steps


# if __name__ == "__main__":
# train()


grid_sbox = 8, add_space_token = False, subsel_type = all, learning_rate = 0.0005, max_iters = 250, loss_type = EMD
Start running basic DDP example on rank 0.
ind_sel_train = (96000,), ind_sel_val = (22400,)
all tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) torch.Size([96000, 12, 8, 8, 8]) torch.Size([22400, 12, 8, 8, 8]) False
block_size = 481, vocab_size = 69, pad_token = 67, max_sentence_length = 482
nembd = 256, nhead = 8, nlayer = 4, nparams = 5, dropout = 0.2
Using flash:  True
Using flash:  True
Using flash:  True
Using flash:  True
number of parameters: 6.86M
Init model and loaded to GPU
nbatches = 93, total train size = 96000


/tmp/ipykernel_2571245/1661427923.py:174: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


In [4]:
# Y


In [ ]:
while True:
    lr = get_lr(iter_num, model=decay_lr_model) if decay_lr else learning_rate
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0 and (rank == 0):
            losses = estimate_loss()
            print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                if iter_num > 0:
                    checkpoint = {
                        'model': model.module.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'iter_num': iter_num,
                        'best_val_loss': best_val_loss,
                        'config': HaloConfig,
                        'lr': lr
                    }
                    print(f"saving checkpoint")
                    # torch.save(checkpoint, f'/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL5_afterfinetune_{ds_type_here}_TEST_model_hres_encdec_ddp_grid_{grid_sbox}_nvocab_{nvocab}_nembed_{n_embd}_nhead_{n_head}_nrandsubsel_{int(nrand_sel_box/(subsamp_ds * ds_fac_here))}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_maxiter_{max_iters}_lr_{learning_rate}.pt')                                 
                    # torch.save(checkpoint, f'/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/FINAL6_nofidfinetune_{ds_type_here}_TEST_model_hres_encdec_ddp_grid_{grid_sbox}_nvocab_{nvocab}_nembed_{n_embd}_nhead_{n_head}_nrandsubsel_{int(nrand_sel_box/(subsamp_ds * ds_fac_here))}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_maxiter_{max_iters}_lr_{learning_rate}.pt')                                 
                    # torch.save(checkpoint, f'/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/iter_save/FINAL2_iter_{iter_num}_{ds_type_here}_TEST_model_hres_encdec_ddp_grid_{grid_sbox}_nvocab_{nvocab}_nembed_{n_embd}_nhead_{n_head}_nrandsubsel_{int(nrand_sel_box/(subsamp_ds * ds_fac_here))}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_maxiter_{max_iters}_lr_{learning_rate}.pt')                                 
                    torch.save(checkpoint, f'/projects/bdne/spandey3/halo_gotham/GOTHAM/model_checkpoints/quijote_halos/iter_save/TESTLOSS_{loss_type}_iter_{iter_num}_{ds_type_here}_TEST_model_hres_encdec_ddp_grid_{grid_sbox}_nvocab_{nvocab}_nembed_{n_embd}_nhead_{n_head}_nrandsubsel_{int(nrand_sel_box/(subsamp_ds * ds_fac_here))}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_maxiter_{max_iters}_lr_{learning_rate}.pt')                                 


    for ji in (range(nbatches)):
        model.require_backward_grad_sync = (ji == nbatches - 1)

        X, Y, MASK, DM, PARAMS = get_batch('train', ji, batch_size)
        with ctx:
            _, loss = model(X, DM, params=PARAMS, maskd=MASK, targets=Y, loss_type=loss_type)
        scaler.scale(loss).backward()   
        torch.cuda.empty_cache() 

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    iter_num += 1
    local_iter_num += 1

    # termination conditions
    if iter_num > max_iters:
        break


# dist.destroy_process_group()

/u/spandey3/gotham/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


step 0: train loss 4.3038, val loss 4.3041


/u/spandey3/gotham/lib/python3.10/site-packages/torch/autograd/graph.py:817: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/MHA.cpp:667.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


step 4: train loss 4.2433, val loss 4.2408
saving checkpoint
step 8: train loss 4.1754, val loss 4.1661
saving checkpoint
step 12: train loss 4.1641, val loss 4.1464
saving checkpoint
step 16: train loss 4.1533, val loss 4.1383
saving checkpoint
step 20: train loss 4.1441, val loss 4.1312
saving checkpoint
step 24: train loss 4.1300, val loss 4.1218
saving checkpoint
step 28: train loss 4.0982, val loss 4.0884
saving checkpoint
step 32: train loss 4.0527, val loss 4.0421
saving checkpoint
step 36: train loss 4.0407, val loss 4.0343
saving checkpoint
step 40: train loss 4.0284, val loss 4.0210
saving checkpoint
step 44: train loss 4.0217, val loss 4.0163
saving checkpoint
step 48: train loss 4.0287, val loss 4.0235
step 52: train loss 4.0412, val loss 4.0328
step 56: train loss 4.0379, val loss 4.0274
step 60: train loss 4.0744, val loss 4.0677
step 64: train loss 4.0256, val loss 4.0202
step 68: train loss 4.0326, val loss 4.0054
saving checkpoint
